# Notebook 04 — Storage Load

**Rubric:** §4.2 Storage

Demonstrates loading all three cleaned sources into Neon Postgres and MongoDB Atlas. Runs schema creation, all three loaders, and verifies row counts.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
print('Working directory:', os.getcwd())

In [ ]:
from src.storage import postgres_setup, mongo_setup

print('Creating Postgres schema...')
postgres_setup.create_all()
print('\nCreating MongoDB indexes...')
mongo_setup.create_all()
print('\nSchema setup complete.')

## Load Parquet Sample (500K rows, stratified)

The full 4M-row parquet expands to ~2 GB in Postgres, exceeding the Neon free-tier 0.5 GB cap. We load a stratified 500K-row sample (`random_state=42`), preserving fraud and churn distributions.

In [ ]:
from src.storage.load_parquet_to_postgres import run as load_parquet
load_parquet()

## Load XML Transactions (28,360 rows)

In [ ]:
from src.storage.load_xml_to_postgres import run as load_xml
load_xml()

## Load JSON Profiles to MongoDB (~375K documents)

In [ ]:
from src.storage.load_json_to_mongo import run as load_json
load_json()

## Verify Counts

In [ ]:
from sqlalchemy import text
from src.config import get_pg_engine, get_mongo_db

engine = get_pg_engine()
db = get_mongo_db()

tables = ['fact_mobile_money_tx', 'fact_xml_transactions', 'dim_wallet_agg_mm', 'dim_wallet_agg_xml']
print('Postgres row counts:')
with engine.connect() as conn:
    for t in tables:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'  {t}: {n:,}')

mongo_count = db.customer_profiles.count_documents({})
print(f'\nMongoDB customer_profiles: {mongo_count:,} documents')

**Expected results:**
- `fact_mobile_money_tx` ≈ 500,000 rows
- `fact_xml_transactions` = 28,360 rows
- `dim_wallet_agg_mm` = number of unique wallets in sample
- `dim_wallet_agg_xml` = 479 wallets
- MongoDB `customer_profiles` ≈ 375,837 documents